In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
bronze = spark.table("workspace.default.bronze_prices")
test = bronze.withColumn("start_utc", F.to_timestamp("time_start"))
test.select("time_start", "start_utc").show(3, truncate=False)

+-------------------------+-------------------+
|time_start               |start_utc          |
+-------------------------+-------------------+
|2026-10-05T00:00:00+02:00|2026-10-04 22:00:00|
|2026-10-05T01:00:00+02:00|2026-10-04 23:00:00|
|2026-10-05T02:00:00+02:00|2026-10-05 00:00:00|
+-------------------------+-------------------+
only showing top 3 rows


In [0]:
silver = (
    bronze
    .withColumn("start_utc", F.to_timestamp("time_start"))
    .withColumn("end_utc", F.to_timestamp("time_end"))
    .select(
        "area",
        "start_utc",
        "end_utc",
        F.col("NOK_per_kWh").alias("price_nok_kwh"),
        F.col("EUR_per_kWh").alias("price_eur_kwh"),
        "source_file",
    )
    .dropDuplicates(["area", "start_utc"])
)
print(silver.count())

1920


In [0]:
silver.write.mode("overwrite").saveAsTable("workspace.default.silver_prices")
spark.table("workspace.default.silver_prices").orderBy("area", "start_utc").show(3, truncate=False)

+----+-------------------+-------------------+-------------+-------------+--------------------------------------------+
|area|start_utc          |end_utc            |price_nok_kwh|price_eur_kwh|source_file                                 |
+----+-------------------+-------------------+-------------+-------------+--------------------------------------------+
|NO1 |2026-10-04 22:00:00|2026-10-04 23:00:00|1.28581      |0.11871      |prices/2026-10-05/prices_2026-10-05_NO1.json|
|NO1 |2026-10-04 23:00:00|2026-10-05 00:00:00|1.22331      |0.11294      |prices/2026-10-05/prices_2026-10-05_NO1.json|
|NO1 |2026-10-05 00:00:00|2026-10-05 01:00:00|1.17988      |0.10893      |prices/2026-10-05/prices_2026-10-05_NO1.json|
+----+-------------------+-------------------+-------------+-------------+--------------------------------------------+
only showing top 3 rows
